In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import warnings


# Paths to your model directories
MODEL_DIRS = {
    'LDM': Path("evaluation_results_THESIS_FINAL_2/generated_samples"),
    'GAN': Path("pix2pix_baseline"),
    'Transformer': Path("transformer_full_val_eval"),
}

In [8]:
def compute_gcnr(real_img, gen_img, mask_img):
    """Calculate gCNR using the correct formula."""
    real_img = np.asarray(real_img)
    gen_img = np.asarray(gen_img)
    mask_img = np.asarray(mask_img)
    
    fg_mask = mask_img > 0
    bg_mask = mask_img == 0
    if np.sum(fg_mask) == 0 or np.sum(bg_mask) == 0:
        return np.nan
    
    fg = gen_img[fg_mask].flatten()
    bg = gen_img[bg_mask].flatten()
    
    mu_fg, mu_bg = np.mean(fg), np.mean(bg)
    std_fg, std_bg = np.std(fg), np.std(bg)
    pooled_noise = np.sqrt(0.5 * (std_fg**2 + std_bg**2))
    if pooled_noise == 0:
        return np.nan
    signal_diff = np.abs(mu_fg - mu_bg)
    gsnr = signal_diff / pooled_noise
    gcnr = 1 - 2 * norm.cdf(-gsnr / 2)
    return gcnr

def to_numpy(x):
    if torch.is_tensor(x):
        return x.cpu().numpy()
    return np.asarray(x)

def load_case(case_name):
    """Load all three model outputs and compute metrics."""
    ldm_file = MODEL_DIRS['LDM'] / f"{case_name}.pt"
    gan_file = MODEL_DIRS['GAN'] / f"{case_name}.pt"
    trans_file = MODEL_DIRS['Transformer'] / f"{case_name}.pt"
    
    ldm_data = torch.load(ldm_file, weights_only=False)
    gan_data = torch.load(gan_file, weights_only=False)
    trans_data = torch.load(trans_file, weights_only=False)
    
    # Extract data
    gt = to_numpy(ldm_data['real_flair'])
    mask = to_numpy(ldm_data['mask'])
    ldm_synth = to_numpy(ldm_data['synthetic_flair'])
    gan_synth = to_numpy(gan_data['synthetic_flair'])
    
    # Handle Transformer key (try common alternatives)
    if 'synthetic_flair' in trans_data:
        trans_synth = to_numpy(trans_data['synthetic_flair'])
    elif 'syn' in trans_data:
        trans_synth = to_numpy(trans_data['syn'])
    elif 'output' in trans_data:
        trans_synth = to_numpy(trans_data['output'])
    else:
        raise KeyError(f"Unknown key for Transformer in {case_name}. Keys: {list(trans_data.keys())}")
    
    # PSNR with dynamic range
    data_range = np.max(gt) - np.min(gt)
    if data_range == 0:
        data_range = 1.0
    
    ldm_psnr = psnr(gt, ldm_synth, data_range=data_range)
    gan_psnr = psnr(gt, gan_synth, data_range=data_range)
    trans_psnr = psnr(gt, trans_synth, data_range=data_range)
    
    ldm_gcnr = compute_gcnr(gt, ldm_synth, mask)
    gan_gcnr = compute_gcnr(gt, gan_synth, mask)
    trans_gcnr = compute_gcnr(gt, trans_synth, mask)
    
    return {
        'case': case_name,
        'gt': gt, 'mask': mask,
        'ldm': ldm_synth, 'gan': gan_synth, 'trans': trans_synth,
        'ldm_psnr': ldm_psnr, 'gan_psnr': gan_psnr, 'trans_psnr': trans_psnr,
        'ldm_gcnr': ldm_gcnr, 'gan_gcnr': gan_gcnr, 'trans_gcnr': trans_gcnr,
    }

In [ ]:
if CSV_PATH.exists():
    print(f"Loading existing CSV from {CSV_PATH}")
    df = pd.read_csv(CSV_PATH)
else:
    print("CSV not found. Running analysis to generate all_metrics.csv...")
    
    ldm_files = sorted(MODEL_DIRS['LDM'].glob("*.pt"))
    all_metrics = []
    
    for f in ldm_files:
        name = f.stem
        gan_file = MODEL_DIRS['GAN'] / f"{name}.pt"
        trans_file = MODEL_DIRS['Transformer'] / f"{name}.pt"
        
        if not (gan_file.exists() and trans_file.exists()):
            continue
        
        try:
            ldm_data = torch.load(f, weights_only=False)
            gan_data = torch.load(gan_file, weights_only=False)
            trans_data = torch.load(trans_file, weights_only=False)
        except Exception as e:
            print(f"Error loading {name}: {e}")
            continue
        
        gt = to_numpy(ldm_data['real_flair'])
        mask = to_numpy(ldm_data['mask'])
        ldm_synth = to_numpy(ldm_data['synthetic_flair'])
        gan_synth = to_numpy(gan_data['synthetic_flair'])
        
        # Transformer key handling
        if 'synthetic_flair' in trans_data:
            trans_synth = to_numpy(trans_data['synthetic_flair'])
        elif 'syn' in trans_data:
            trans_synth = to_numpy(trans_data['syn'])
        elif 'output' in trans_data:
            trans_synth = to_numpy(trans_data['output'])
        else:
            continue
        
        if np.sum(mask > 0) == 0:
            continue
        
        data_range = np.max(gt) - np.min(gt)
        if data_range == 0:
            data_range = 1.0
        
        ldm_psnr = psnr(gt, ldm_synth, data_range=data_range)
        gan_psnr = psnr(gt, gan_synth, data_range=data_range)
        trans_psnr = psnr(gt, trans_synth, data_range=data_range)
        
        ldm_gcnr = compute_gcnr(gt, ldm_synth, mask)
        gan_gcnr = compute_gcnr(gt, gan_synth, mask)
        trans_gcnr = compute_gcnr(gt, trans_synth, mask)
        
        if any(np.isnan(v) for v in [ldm_gcnr, gan_gcnr, trans_gcnr]):
            continue
        
        all_metrics.append({
            'case': name,
            'ldm_psnr': ldm_psnr, 'gan_psnr': gan_psnr, 'trans_psnr': trans_psnr,
            'ldm_gcnr': ldm_gcnr, 'gan_gcnr': gan_gcnr, 'trans_gcnr': trans_gcnr,
        })
    
    df = pd.DataFrame(all_metrics)
    df.to_csv(CSV_PATH, index=False)
    print(f"Saved CSV with {len(df)} valid cases to {CSV_PATH}")

print(f"\nTotal valid cases: {len(df)}")
df.head()

In [ ]:
# Case Type 1: LDM has the best gCNR (higher than both GAN and Transformer)
ldm_best_df = df[
    (df['ldm_gcnr'] > df['gan_gcnr']) & 
    (df['ldm_gcnr'] > df['trans_gcnr'])
].copy()

# Sort by how much better LDM is (sum of differences)
ldm_best_df['ldm_advantage'] = (ldm_best_df['ldm_gcnr'] - ldm_best_df['gan_gcnr']) + \
                                (ldm_best_df['ldm_gcnr'] - ldm_best_df['trans_gcnr'])
ldm_best_df = ldm_best_df.sort_values('ldm_advantage', ascending=False)

print(f"Case Type 1: LDM best gCNR → {len(ldm_best_df)} cases")
ldm_best_df[['case', 'ldm_psnr', 'gan_psnr', 'trans_psnr', 'ldm_gcnr', 'gan_gcnr', 'trans_gcnr']].head(5)

In [ ]:
# Case Type 2: GAN has better PSNR but worse gCNR (the trade-off)
tradeoff_df = df[
    (df['gan_psnr'] > df['ldm_psnr']) & 
    (df['gan_gcnr'] < df['ldm_gcnr'])
].copy()

# Sort by combined metric: PSNR gain + gCNR loss
tradeoff_df['tradeoff_score'] = (tradeoff_df['gan_psnr'] - tradeoff_df['ldm_psnr']) + \
                                 (tradeoff_df['ldm_gcnr'] - tradeoff_df['gan_gcnr'])
tradeoff_df = tradeoff_df.sort_values('tradeoff_score', ascending=False)

print(f"Case Type 2: GAN PSNR better but gCNR worse → {len(tradeoff_df)} cases")
tradeoff_df[['case', 'ldm_psnr', 'gan_psnr', 'trans_psnr', 'ldm_gcnr', 'gan_gcnr', 'trans_gcnr']].head(5)

In [14]:
def plot_case(data, title=None, figsize=(16, 4)):
    """Display a 4-column montage for a single case."""
    cols = 4
    fig, axes = plt.subplots(1, cols, figsize=figsize)
    
    titles = ['Ground Truth + Mask', 'LDM', 'GAN', 'Transformer']
    images = [data['gt'], data['ldm'], data['gan'], data['trans']]
    psnrs = [None, data['ldm_psnr'], data['gan_psnr'], data['trans_psnr']]
    gcnrs = [None, data['ldm_gcnr'], data['gan_gcnr'], data['trans_gcnr']]
    
    for ax, img, t, ps, gc in zip(axes, images, titles, psnrs, gcnrs):
        ax.imshow(img, cmap='gray')
        if t == 'Ground Truth + Mask':
            # Show mask boundary in red
            mask_boundary = find_boundaries(data['mask'] > 0, mode='outer')
            ax.imshow(mask_boundary, cmap='Reds', alpha=0.5)
        ax.set_title(t, fontsize=12)
        if ps is not None:
            ax.set_xlabel(f"PSNR: {ps:.2f} dB\ngCNR: {gc:.4f}", fontsize=10)
        ax.axis('off')
    
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Get top 5 cases from each category
top_ldm_best = ldm_best_df.head(5)['case'].tolist()
top_tradeoff = tradeoff_df.head(5)['case'].tolist()

print("="*60)
print("📊 CASE TYPE 1: LDM has the best gCNR (Clinical Contrast)")
print("="*60)
for case_name in top_ldm_best:
    data = load_case(case_name)
    plot_case(data, title=f"Case: {case_name}", figsize=(16, 4))

print("="*60)
print("📊 CASE TYPE 2: GAN has better PSNR but worse gCNR (Trade-off)")
print("="*60)
for case_name in top_tradeoff:
    data = load_case(case_name)
    plot_case(data, title=f"Case: {case_name}", figsize=(16, 4))